In [4]:
import pynetlogo
import pandas as pd
import itertools
from pathlib import Path

# 1. Explicitly point to the jvm.dll file inside NetLogo's directory
jvm_path = "C:/Program Files/NetLogo 6.4.0/runtime/bin/server/jvm.dll"

netlogo = pynetlogo.NetLogoLink(
    gui=False,
    netlogo_home="C:/Program Files/NetLogo 6.4.0",
    jvm_path=jvm_path 
)

model_path = "C:/Users/15177459/Desktop/netlogo/models/survey_derived.nlogo"

netlogo.load_model(model_path)
netlogo.command("setup")

print("People:", netlogo.report("count people"))
print("Places:", netlogo.report("count places"))
print("Ticks:", netlogo.report("ticks"))

People: 300.0
Places: 60.0
Ticks: 0.0


In [ ]:
parameter_grid = {
    "number-of-people": [100, 200, 300],
    "baseline-stop-probability": [18, 30, 40],
    "third-place-search-radius": [1, 2, 5],
    "route-third-place-sample-size": [50],
    "rigid-dwell-time": [30],
    "medium-dwell-time": [45],
    "flexible-dwell-time": [60],
    "encounter-odds-increment": [0.05, 0.10, 0.20],
    "encounter-weight": [1, 5, 8],
    "social-feedback?": [True],
}

reporters = [
    "average-route-third-place-options",
    "people-with-route-third-place-options",
    "visits-per-person",
    "average-stop-probability",
    "average-social-encounters",
    "average-social-odds-ratio",
    "total-third-place-visits",
    "total-co-presence",
    "co-presence-per-visit",
    "places-with-co-presence",
    "share-places-visited",
    "rigid-visits-per-person",
    "medium-visits-per-person",
    "flexible-visits-per-person"
]

results = []

keys = list(parameter_grid.keys())
values = list(parameter_grid.values())

run_id = 0

for combination in itertools.product(*values):
    params = dict(zip(keys, combination))
    
    for seed in range(1, 6):  # start with 5 repetitions
        run_id += 1
        
        netlogo.command(f"random-seed {seed}")
        
        for parameter, value in params.items():
            if isinstance(value, bool):
                value = "true" if value else "false"
            netlogo.command(f"set {parameter} {value}")
        
        # Link rigid/flexible dwell time to medium dwell time
        medium_dwell = params["medium-dwell-time"]
        netlogo.command(f"set rigid-dwell-time {max(5, medium_dwell * 0.5)}")
        netlogo.command(f"set flexible-dwell-time {medium_dwell * 1.5}")
        
        netlogo.command("setup")
        netlogo.command("repeat 1000 [ go ]")
        
        row = {
            "run_id": run_id,
            "seed": seed,
            **params,
        }
        
        for reporter in reporters:
            row[reporter] = netlogo.report(reporter)
        
        results.append(row)

df = pd.DataFrame(results)
df.head(20)

In [ ]:
output_path = "/Users/tonyvo/Desktop/Thesis/netlogo/models/sweep_results_test.csv"

df.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")
print(df.shape)

Saved to: /Users/tonyvo/Desktop/Thesis/netlogo/models/sweep_results_test.csv
(720, 17)


In [ ]:
summary = (
    df.groupby([
        "number-of-people",
        "baseline-stop-probability",
        "medium-dwell-time",
        "encounter-odds-increment",
        "encounter-weight"
    ])
    [[
        "visits-per-person",
        "average-stop-probability",
        "average-social-encounters",
        "average-social-odds-ratio",
        "total-co-presence",
        "co-presence-per-visit"
    ]]
    .mean()
    .reset_index()
)

summary.sort_values(
    by=["average-social-odds-ratio", "average-social-encounters"],
    ascending=False
).head(20)

,number-of-people,baseline-stop-probability,medium-dwell-time,encounter-odds-increment,encounter-weight,visits-per-person,average-stop-probability,average-social-encounters,average-social-odds-ratio,total-co-presence,co-presence-per-visit
143,200,30,60,0.20,8,1.689,34.336697,9.792,1.9174,6329.6,18.661408
142,200,30,60,0.20,5,1.620,33.008827,5.670,1.7874,5672.2,17.471221
131,200,30,45,0.20,8,1.642,32.618829,7.360,1.7728,3718.8,11.310743
139,200,30,60,0.10,8,1.614,32.101007,8.976,1.7012,5810.0,18.008268
130,200,30,45,0.20,5,1.593,30.996637,4.170,1.6202,3299.2,10.348343
71,100,30,60,0.20,8,1.642,30.476906,4.480,1.6144,1478.0,9.020747
141,200,30,60,0.20,3,1.607,30.895743,3.396,1.5928,5600.6,17.440035
127,200,30,45,0.10,8,1.580,29.705309,5.888,1.5086,3082.6,9.743054
138,200,30,60,0.10,5,1.572,29.864410,5.400,1.5063,5594.4,17.789292
129,200,30,45,0.20,3,1.617,29.506706,2.604,1.4824,3570.6,11.024886


In [ ]:
weight_effect = (
    df.groupby("encounter-weight")
    [[
        "visits-per-person",
        "average-stop-probability",
        "average-social-encounters",
        "average-social-odds-ratio",
        "total-co-presence",
        "co-presence-per-visit"
    ]]
    .mean()
    .reset_index()
)

weight_effect

,encounter-weight,visits-per-person,average-stop-probability,average-social-encounters,average-social-odds-ratio,total-co-presence,co-presence-per-visit
0,1,1.284167,18.599552,0.360000,1.042736,1345.066667,6.165572
1,3,1.308722,19.703414,1.110667,1.127508,1368.461111,6.160836
2,5,1.326250,20.545353,1.886111,1.200378,1381.527778,6.142976
3,8,1.342278,21.411438,3.114667,1.280528,1439.027778,6.254880


In [ ]:
feedback_strength = (
    df.groupby([
        "encounter-weight",
        "encounter-odds-increment"
    ])
    [[
        "visits-per-person",
        "average-stop-probability",
        "average-social-encounters",
        "average-social-odds-ratio",
        "total-co-presence",
        "co-presence-per-visit"
    ]]
    .mean()
    .reset_index()
)

feedback_strength.sort_values(
    by=["encounter-weight", "encounter-odds-increment"]
)

,encounter-weight,encounter-odds-increment,visits-per-person,average-stop-probability,average-social-encounters,average-social-odds-ratio,total-co-presence,co-presence-per-visit
0,1,0.05,1.269250,18.217277,0.340833,1.017042,1290.016667,6.040131
1,1,0.10,1.285917,18.522108,0.366667,1.036667,1371.733333,6.257678
2,1,0.20,1.297333,19.059272,0.372500,1.074500,1373.450000,6.198906
3,3,0.05,1.289417,18.775795,1.086000,1.054300,1351.433333,6.232924
4,3,0.10,1.304667,19.506951,1.089500,1.108725,1335.150000,6.042208
5,3,0.20,1.332083,20.827495,1.156500,1.219500,1418.800000,6.207375
6,5,0.05,1.297500,19.261843,1.795833,1.089758,1333.400000,6.082039
7,5,0.10,1.327417,20.372760,1.852500,1.180308,1369.483333,6.069655
8,5,0.20,1.353833,22.001456,2.010000,1.331067,1441.700000,6.277233
9,8,0.05,1.308333,19.999009,3.005333,1.148167,1384.266667,6.123212
